In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import pickle

from nteprsm import utils 
from settings import DATA_DIR
file_path = DATA_DIR/'data_ingestion/parsed_data/kb2017/nj2/quality.csv'
fit_obj_path = DATA_DIR/'model_output/kb2017/fit_seasonality_nj2_quality.pkl'
entrycode2name_path = DATA_DIR/'model_output/entrycode2name.json'

def map_name2code(datahandler, column_name, code_column_name, invert=False):
    """Retrieves a dictionary mapping names to codes from specified columns."""
    name2code = dict(datahandler.model_data.groupby(column_name)[code_column_name].first())
    if invert:
        name2code = {v: k for k, v in name2code.items()}
    return name2code

def extract_time_effect(datahandler: "utils.DataHandler", fit: "stan fit object") -> pd.DataFrame:
    """Extract and format the time effect from a fitted model object."""
    raing_event2doy = datahandler.model_data[['adj_time_of_year', 'rating_event_code']].drop_duplicates(
        ).sort_values(by='rating_event_code')
    time_effect = pd.DataFrame(
        fit.stan_variable("time_effect").mean(axis=0),
        columns=raing_event2doy['adj_time_of_year'].values
    )
    
    time_effect.index = time_effect.index
    time_effect = time_effect.T.sort_index()
    time_effect.index.name = 'adj_time_of_year'
    return time_effect

In [13]:
with open(fit_obj_path, 'rb') as file:
    fit = pickle.load(file)

In [6]:
datahandler = utils.DataHandler(filepath=file_path)
datahandler.preprocess_data()
datahandler.generate_stan_data()

2025-07-01 02:38:28,810 - NtepRsm - DEBUG - Logging is already configured.
2025-07-01 02:38:28,821 - NtepRsm - INFO - Raw data successfully loaded.
/Users/henryqu/Documents/GitHub/nteprsm/nteprsm/utils.py:244: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(df["date"], errors="coerce")
2025-07-01 02:38:28,883 - NtepRsm - INFO - Preprocessing complete. Model data ready.
2025-07-01 02:38:28,888 - NtepRsm - INFO - Stan data dictionary created.


In [35]:
name2code = map_name2code(datahandler, 'entry_name', 'entry_code', invert=True)

In [51]:
colors = ['royalblue', 'mediumseagreen', 'deepskyblue', 'crimson']
entries = [7, 19, 29, 56]

## params
ci = 0.95  # Credible interval

# Create a single figure
fig = go.Figure()

# time_effect_data_points
data_points = extract_time_effect(datahandler, fit)
data_points_time = data_points.index.values

for i, entry in enumerate(entries):
    entry_name = name2code[entry]
    legend_group = f"Entry {entry_name}"    
    samples = fit.pred_time_effect[:, entry, :]  # Shape: (n_samples, n_time_points)
    
    # Compute statistics
    time_points = np.arange(samples.shape[1])+1  # Time indices
    mean_values = np.mean(samples, axis=0)  # Mean over samples
    lower_bound = np.percentile(samples, 100 * (1 - ci) / 2, axis=0)  # 2.5th percentile
    upper_bound = np.percentile(samples, 100 * (ci + (1 - ci) / 2), axis=0)  # 97.5th percentile

    # Add data point
    fig.add_trace(go.Scatter(
        x=data_points_time*100, y=data_points[entry],
        mode='markers',
        marker=dict(
            symbol='circle',  # Set the marker shape to "X"
            color=colors[i],  # Keep the color
            size=10  # Optionally, set the size of the marker
        ),
        name=f'{entry_name} Mean',
        legendgroup=legend_group,
        showlegend=False,
    ))

    # Add mean line
    fig.add_trace(go.Scatter(
        x=time_points, y=mean_values,
        mode='lines',
        line=dict(color=colors[i], width=2),
        name=f'{entry_name}',
        legendgroup=legend_group,
    ))
    
    # Add 95% Credible Interval (Shaded Region)
    fig.add_trace(go.Scatter(
        x=time_points.tolist() + time_points[::-1].tolist(),
        y=upper_bound.tolist() + lower_bound[::-1].tolist(),
        fill='toself',
        fillcolor=colors[i],
        opacity=0.2,
        line=dict(color='rgba(255,255,255,0)'),
        name=f'{entry_name} 95% CI',
        legendgroup=legend_group,  # Grouping
        showlegend=False,
    ))
    
# Compute correct tick positions based on actual day of the year
days_in_month = np.array([0, 31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])  # Non-leap year
cumulative_days = np.cumsum(days_in_month)+1  # Cumulative sum to get end of each month
month_positions = 100*cumulative_days / 365  # Normalize to range 0-100
month_positions[0] += 0.8

# Generate month labels
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig.update_layout(
    xaxis=dict(
        tickmode="array",
        tickvals=month_positions,  # Correctly spaced tick positions
        ticktext=month_labels,  # Month names
        title="Time of Year"
    ),
    yaxis_title="Seasonality (Latent Scale)",
    title=f"Turfgrass Seasonality, {ci} credible interval",
    template="ggplot2",
    legend=dict(
        title="Entries",
        orientation="h",
        yanchor="bottom",
        y=-0.3,
        xanchor="center",
        x=0.5
    )
) 

# needed to adjust figure size here with static renderer for plotly
pio.kaleido.scope.default_width = 1200
pio.kaleido.scope.default_height = 400
fig.show()